In [4]:
import os, shutil, random, pathlib

original_dir = pathlib.Path("./dataset")
base_dir = pathlib.Path("./dataset_split")

splits = ["train", "validation", "test"]

for split in splits:
    for class_name in os.listdir(original_dir):
        os.makedirs(base_dir / split / class_name, exist_ok=True)

for class_name in os.listdir(original_dir):
    images = list((original_dir / class_name).glob("*"))
    random.shuffle(images)
    
    n = len(images)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    for img in train_images:
        shutil.copy(img, base_dir / "train" / class_name / img.name)
        
    for img in val_images:
        shutil.copy(img, base_dir / "validation" / class_name / img.name)
        
    for img in test_images:
        shutil.copy(img, base_dir / "test" / class_name / img.name)

print("Dataset jaettu train/validation/test")

Dataset jaettu train/validation/test


In [5]:
import matplotlib.pyplot as plt
import keras
from keras.utils import image_dataset_from_directory
from keras import layers

train_dataset = image_dataset_from_directory(
    "./dataset_split/train",
    image_size=(180, 180),
    batch_size=32
)

validation_dataset = image_dataset_from_directory(
    "./dataset_split/validation",
    image_size=(180, 180),
    batch_size=32
)

test_dataset = image_dataset_from_directory(
    "./dataset_split/test",
    image_size=(180, 180),
    batch_size=32
)

class_names = train_dataset.class_names
num_classes = len(class_names)

print("Luokat: ", class_names)
print("Luokkien määrä: ", num_classes)


Found 139 files belonging to 3 classes.


Found 42 files belonging to 3 classes.
Found 46 files belonging to 3 classes.
Luokat:  ['autot', 'laivat', 'lentokoneet']
Luokkien määrä:  3


In [6]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.2)
])

In [7]:
x = layers.Conv2D(64, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)

x = layers.Conv2D(128, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)

x = layers.Conv2D(256, 3, activation="relu")(x)
x = layers.MaxPooling2D(2)(x)

x = layers.Conv2D(256, 3, activation="relu")(x)

x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(num_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.summary()

NameError: name 'x' is not defined

In [ ]:
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="model1_cnn.keras",
        save_best_only=True,
        monitor="val_loss"
    )
]

history = model.fit(
    train_dataset,
    epochs=15,
    validation_data=validation_dataset,
    callbacks=callbacks
)

In [ ]:
accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]

loss = history.history["loss"]
val_loss = history.history["val_loss"]

epochs = range(1, len(accuracy) + 1)

plt.plot(epochs, accuracy, label="Training accuracy")
plt.plot(epochs, val_accuracy, label="Validation accuracy")
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(epochs, loss, label="Training loss")
plt.plot(epochs, val_loss, label="Validation loss")
plt.legend()
plt.title("Loss")
plt.show()

In [ ]:
test_model = keras.models.load_model("model1_cnn.keras")

test_loss, test_acc = test_model.evaluate(test_dataset)

print(f"Test accuracy: {test_acc:.3f}")